In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

os.listdir('/content/drive/MyDrive/anemia_detection/test_dataset')

['palm_test.zip', 'archive.zip']

In [ ]:
!unzip "/content/drive/MyDrive/anemia_detection/test_dataset/palm_test.zip"


Archive:  /content/drive/MyDrive/anemia_detection/test_dataset/palm_test.zip
replace Palm/Anemic-260 (10).png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!pip install catboost

In [ ]:
import os
import glob
import cv2
import numpy as np
import joblib
import pandas as pd

# ----------------------------
# 1. Dosya yolları
# ----------------------------
palm_folder = "/content/Palm"  # Palm veri seti
model_folder = "/content/drive/MyDrive/anemia_detection/saved_models/palm_models"

# Anemic ve Non-anemic dosyalarını al
anemic_files = glob.glob(os.path.join(palm_folder, "Anemic*.png")) + \
               glob.glob(os.path.join(palm_folder, "Anemic*.jpg"))

non_anemic_files = glob.glob(os.path.join(palm_folder, "Non-Anemic*.png")) + \
                   glob.glob(os.path.join(palm_folder, "Non-Anemic*.jpg")) + \
                   glob.glob(os.path.join(palm_folder, "Non-anemic*.png")) + \
                   glob.glob(os.path.join(palm_folder, "Non-anemic*.jpg"))

all_files = anemic_files + non_anemic_files

# Sınıf etiketleri (ground truth)
labels = [1]*len(anemic_files) + [0]*len(non_anemic_files)

In [ ]:
# ----------------------------
# 2. Özellik çıkarımı (LAB a,b ve RGB G ortalaması)
# ----------------------------
features = []
labels_new = []

for f, label in zip(all_files, labels):
    img = cv2.imread(f)
    if img is None:
        continue

    # RGB formatına çevir
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # LAB renk uzayına çevir
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab)

    # Özellikler: a kanalı ortalaması, b kanalı ortalaması, G kanalı ortalaması
    mean_a = np.mean(a)
    mean_b = np.mean(b)
    mean_g = np.mean(img_rgb[:, :, 1])

    features.append([mean_a, mean_b, mean_g])
    labels_new.append(label)  # 1=Anemic, 0=Non-Anemic

# Numpy dizilerine çevir
X_new = np.array(features, dtype=np.float32)
y_true = np.array(labels_new, dtype=np.int32)

print("Yeni veri sayısı:", len(X_new))
print("X_new shape:", X_new.shape)
print("y_true shape:", y_true.shape)
print("Sınıf dağılımı:", np.bincount(y_true))

Yeni veri sayısı: 4260
X_new shape: (4260, 3)
y_true shape: (4260,)
Sınıf dağılımı: [1698 2562]


In [ ]:
model_files = glob.glob(os.path.join(model_folder, "*.joblib"))

results = []

for mf in model_files:
    model_name = os.path.basename(mf).replace(".joblib","").replace("_"," ")
    model = joblib.load(mf)

    # Tahmin
    y_pred = model.predict(X_new)
    acc = np.mean(y_pred == y_true)
    print(f"{model_name}: Doğruluk (Accuracy) = {acc:.4f}")

    results.append({
        "Model": model_name,
        "Accuracy": acc
    })

# Özet tablo
results_df = pd.DataFrame(results)
print("\nÖzet:")
print(results_df)

Naive Bayes: Doğruluk (Accuracy) = 0.5915
Decision Tree: Doğruluk (Accuracy) = 0.9972
HistGradientBoosting: Doğruluk (Accuracy) = 0.9986


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM model: Doğruluk (Accuracy) = 0.9986
Gradient Boosting: Doğruluk (Accuracy) = 0.9986
SVM: Doğruluk (Accuracy) = 0.9986
CatBoost: Doğruluk (Accuracy) = 0.9986
AdaBoost: Doğruluk (Accuracy) = 0.9986
KNN: Doğruluk (Accuracy) = 0.9972
XGBoost: Doğruluk (Accuracy) = 0.9986

Özet:
                  Model  Accuracy
0           Naive Bayes  0.591549
1         Decision Tree  0.997183
2  HistGradientBoosting  0.998592
3        LightGBM model  0.998592
4     Gradient Boosting  0.998592
5                   SVM  0.998592
6              CatBoost  0.998592
7              AdaBoost  0.998592
8                   KNN  0.997183
9               XGBoost  0.998592
